In [1]:
import os
import subprocess
from pathlib import Path
import numpy as np
import nibabel as nib
from pathlib import Path
import psutil
import scipy.io as sio
from scipy.stats import spearmanr, pearsonr, kendalltau
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add Connectome Workbench to PATH
os.environ['PATH'] = '/opt/workbench/bin_linux64:' + os.environ['PATH']

# Glasser RSA boilerplate

In [2]:
# ============================================================================
# TIMING AND SEGMENTATION FUNCTIONS
# ============================================================================

def average_2s_bins_delay(X, TR, onset_sec, duration_sec, bin_sec=2.0, delay_sec=4.0):
    """
    Extract and bin fMRI data with BOLD delay compensation.
    
    Parameters:
    -----------
    X : np.ndarray
        fMRI data (n_vertices, n_timepoints)
    TR : float
        Repetition time in seconds
    onset_sec : float
        Stimulus onset in seconds
    duration_sec : float
        Stimulus duration in seconds
    bin_sec : float
        Bin size in seconds (default 2.0)
    delay_sec : float
        BOLD delay in seconds (default 4.0)

    Returns:
    --------
    Y : np.ndarray
        Binned data (n_vertices, n_bins)
    """
    n_vertices, n_timepoints = X.shape

    # Apply delay to the stimulus window
    onset_d = onset_sec + delay_sec
    end_d = onset_sec + duration_sec + delay_sec

    # Convert to timepoint indices
    start_tp = int(np.floor(onset_d / TR))
    end_tp = int(np.ceil(end_d / TR))

    # Clip to valid range
    start_tp = max(start_tp, 0)
    end_tp = min(end_tp, n_timepoints)

    if end_tp <= start_tp:
        return np.zeros((n_vertices, 0), dtype=X.dtype)

    segment = X[:, start_tp:end_tp]

    # Timepoints per bin
    tp_per_bin = int(np.round(bin_sec / TR))
    if tp_per_bin < 1:
        raise ValueError(f"bin_sec/TR is < 1 (bin_sec={bin_sec}, TR={TR})")

    # Drop leftover timepoints that don't fill a full bin
    n_bins = segment.shape[1] // tp_per_bin
    segment = segment[:, :n_bins * tp_per_bin]

    if n_bins == 0:
        return np.zeros((n_vertices, 0), dtype=X.dtype)

    Y = segment.reshape(n_vertices, n_bins, tp_per_bin).mean(axis=2)
    return Y


def process_all_video_segments(fmri_data, timing_df, TR=1.0, bin_sec=2.0, delay_sec=4.0):
    """
    Process all video segments according to timing file.
    
    Parameters:
    -----------
    fmri_data : np.ndarray
        Full fMRI data (n_vertices, n_timepoints)
    timing_df : pd.DataFrame
        DataFrame with columns: video_id, onset_sec, duration_sec
    TR : float
        Repetition time in seconds
    bin_sec : float
        Bin size in seconds
    delay_sec : float
        BOLD delay in seconds
    
    Returns:
    --------
    processed_fmri : np.ndarray
        Concatenated binned data (n_vertices, total_bins)
    """
    segments = []
    
    for _, row in timing_df.iterrows():
        binned = average_2s_bins_delay(
            fmri_data,
            TR=TR,
            onset_sec=row["onset_sec"],
            duration_sec=row["duration_sec"],
            bin_sec=bin_sec,
            delay_sec=delay_sec
        )
        segments.append(binned)
        print(f"  Video {row['video_id']}: {binned.shape[1]} bins")
    
    processed_fmri = np.concatenate(segments, axis=1)
    return processed_fmri

In [3]:
def load_glasser_parcellation(label_file_left, label_file_right):
    """
    Load Glasser parcellation from separate GIFTI label files.
    
    Parameters:
    -----------
    label_file_left : str
        Path to left hemisphere .label.gii file
    label_file_right : str
        Path to right hemisphere .label.gii file
    
    Returns:
    --------
    left_labels : np.ndarray
        Left hemisphere parcel IDs (n_vertices,)
    right_labels : np.ndarray
        Right hemisphere parcel IDs (n_vertices,)
    left_info : dict
        Left hemisphere parcel info {id: name}
    right_info : dict
        Right hemisphere parcel info {id: name}
    """
    print(f"Loading Glasser parcellation from separate hemisphere files...")
    
    # Load left hemisphere
    print(f"\nLeft hemisphere: {label_file_left}")
    gii_left = nib.load(label_file_left)
    left_labels = gii_left.darrays[0].data.astype(int)
    
    # Extract label names from label table
    left_info = {}
    if hasattr(gii_left, 'labeltable') and gii_left.labeltable is not None:
        # labeltable.labels is a LIST of GiftiLabel objects
        for label_obj in gii_left.labeltable.labels:
            key = label_obj.key
            label_name = label_obj.label
            if key > 0:  # Skip background (0)
                left_info[key] = label_name
    
    # Load right hemisphere
    print(f"Right hemisphere: {label_file_right}")
    gii_right = nib.load(label_file_right)
    right_labels = gii_right.darrays[0].data.astype(int)
    
    # Extract label names from label table
    right_info = {}
    if hasattr(gii_right, 'labeltable') and gii_right.labeltable is not None:
        # labeltable.labels is a LIST of GiftiLabel objects
        for label_obj in gii_right.labeltable.labels:
            key = label_obj.key
            label_name = label_obj.label
            if key > 0:  # Skip background (0)
                right_info[key] = label_name
    
    # Get statistics
    left_unique = np.unique(left_labels)
    left_unique = left_unique[left_unique > 0]
    right_unique = np.unique(right_labels)
    right_unique = right_unique[right_unique > 0]
    
    print(f"\nLeft hemisphere: {len(left_unique)} parcels")
    print(f"  Parcel IDs range: {left_unique.min()} to {left_unique.max()}")
    print(f"  Total vertices: {len(left_labels)}")
    
    print(f"\nRight hemisphere: {len(right_unique)} parcels")
    print(f"  Parcel IDs range: {right_unique.min()} to {right_unique.max()}")
    print(f"  Total vertices: {len(right_labels)}")
    
    return left_labels, right_labels, left_info, right_info

In [ ]:
# ============================================================================
# MODEL RDM AND RSA FUNCTIONS
# ============================================================================

def load_model_rdms(model_file):
    """
    Load pre-computed model RDMs or embeddings. If embeddings, compute RDM.
    
    Parameters:
    -----------
    model_file : str
        Path to model data (.npy/.npz)
    
    Returns:
    --------
    model_rdm : np.ndarray
        RDM for the model (n_timepoints, n_timepoints)
    """
    print(f"Loading model data from {model_file}...")
    
    model_data = np.load(model_file)
    
    if model_data.ndim == 2:
        if model_data.shape[0] == model_data.shape[1]:
            print(f"  Loaded pre-computed RDM: {model_data.shape}")
            return model_data
        else:
            print(f"  Computing RDM from embeddings: {model_data.shape}")
            print(f"  Using correlation distance for model RDM ")
            rdm = squareform(pdist(model_data, metric='correlation'))
            print(f"  RDM computed with shape: {rdm.shape}")
            return rdm
    else:
        raise ValueError(f"Unexpected model data shape: {model_data.shape}")


def save_as_gifti_metric(data, output_path, metric_name="metric"):
    """
    Save numpy array as GIFTI metric file for Workbench visualization.
    
    Parameters:
    -----------
    data : np.ndarray
        1D array (n_vertices,)
    output_path : str
        Output .func.gii path
    metric_name : str
        Name for the metric
    """
    darray = nib.gifti.GiftiDataArray(
        data=data.astype(np.float32),
        intent='NIFTI_INTENT_NONE',
        datatype='NIFTI_TYPE_FLOAT32'
    )
    
    meta = nib.gifti.GiftiMetaData()
    meta['Name'] = metric_name
    darray.meta = meta
    
    gii = nib.GiftiImage(darrays=[darray])
    nib.save(gii, output_path)
    print(f"Saved GIFTI metric: {output_path}")


def compute_rsa_map(fmri_data, parcel_labels, parcel_info, model_rdm,
                    output_path=None, method='spearman',
                    save_pvalues=True, min_variance=1e-10):
    """
    Compute RSA correlation map for all Glasser parcels.
    
    Parameters:
    -----------
    fmri_data : np.ndarray
        fMRI data (n_vertices, n_timepoints)
    parcel_labels : np.ndarray
        Parcel labels for each vertex (n_vertices,)
    parcel_info : dict
        Dictionary mapping parcel ID to parcel name
    model_rdm : np.ndarray or str
        Either pre-computed RDM or path to model file
    output_path : str, optional
        Path to save correlation map
    method : str
        'spearman', 'pearson', or 'kendall' for RSA comparison
    save_pvalues : bool
        Whether to also save p-values
    min_variance : float
        Minimum variance threshold to skip constant parcels
    
    Returns:
    --------
    rsa_correlations : np.ndarray
        Correlation values (n_vertices,) - 0 for background
    rsa_pvalues : np.ndarray (optional)
        P-values (n_vertices,) - 1 for background
    """
    n_vertices, n_timepoints = fmri_data.shape
    print(f"fMRI data shape: {n_vertices} vertices × {n_timepoints} timepoints")
    
    # Load model RDM if path provided
    if isinstance(model_rdm, (str, Path)):
        model_rdm = load_model_rdms(model_rdm)
    
    print(f"Model RDM shape: {model_rdm.shape}")
    
    # Verify model RDM matches timepoints
    if model_rdm.shape[0] != n_timepoints:
        raise ValueError(
            f"Model RDM timepoints ({model_rdm.shape[0]}) doesn't match "
            f"fMRI timepoints ({n_timepoints})"
        )
    
    # Vectorize model RDM once (upper triangle)
    triu_indices = np.triu_indices(n_timepoints, k=1)
    model_rdm_vec = model_rdm[triu_indices]
    
    print(f"\nModel RDM statistics:")
    print(f"  Vectorized: {len(model_rdm_vec)} unique pairs")
    print(f"  Range: [{model_rdm_vec.min():.4f}, {model_rdm_vec.max():.4f}]")
    print(f"  Mean: {model_rdm_vec.mean():.4f}")
    print(f"  Std: {model_rdm_vec.std():.4f}")
    
    # Check if model RDM has variance
    if model_rdm_vec.std() < min_variance:
        raise ValueError("Model RDM has zero variance! Cannot compute correlations.")
    
    # Get unique parcels (excluding background = 0)
    unique_parcels = np.unique(parcel_labels)
    unique_parcels = unique_parcels[unique_parcels > 0]
    
    print(f"\nComputing RSA for {len(unique_parcels)} parcels...")
    print(f"RSA comparison method: {method}")
    
    # Initialize output arrays
    rsa_correlations = np.zeros(n_vertices, dtype=np.float32)
    if save_pvalues:
        rsa_pvalues = np.ones(n_vertices, dtype=np.float32)
    
    # Track statistics
    n_computed = 0
    n_skipped = 0
    n_failed = 0
    
    # Store per-parcel results
    parcel_results_list = []
    
    # Main loop - compute RSA for each parcel
    for i, parcel_id in enumerate(unique_parcels):
        if (i + 1) % 50 == 0:
            print(f"Progress: {i+1}/{len(unique_parcels)} ({100*(i+1)/len(unique_parcels):.1f}%) | "
                  f"Computed: {n_computed}, Skipped: {n_skipped}, Failed: {n_failed}")
        
        try:
            # Get parcel name
            parcel_name = parcel_info.get(parcel_id, f"Parcel_{parcel_id}")
            
            # Get parcel mask and activations
            parcel_mask = parcel_labels == parcel_id
            n_parcel_vertices = parcel_mask.sum()
            
            if n_parcel_vertices == 0:
                n_skipped += 1
                continue
            
            # Extract parcel activations (n_timepoints, n_vertices_in_parcel)
            parcel_activations = fmri_data[parcel_mask, :].T
            
            # Check for sufficient variance
            if parcel_activations.std() < min_variance:
                n_skipped += 1
                continue
            
            # Compute brain RDM using correlation distance
            parcel_rdm = squareform(pdist(parcel_activations, metric='correlation'))
            
            # Vectorize parcel RDM
            parcel_rdm_vec = parcel_rdm[triu_indices]
            
            # Check if parcel RDM has variance
            if parcel_rdm_vec.std() < min_variance:
                n_skipped += 1
                continue
            
            # Compute correlation with model RDM
            if method == 'spearman':
                corr, pval = spearmanr(parcel_rdm_vec, model_rdm_vec, nan_policy='omit')
            elif method == 'pearson':
                corr, pval = pearsonr(parcel_rdm_vec, model_rdm_vec)
            elif method == 'kendall':
                corr, pval = kendalltau(parcel_rdm_vec, model_rdm_vec, nan_policy='omit')
            else:
                raise ValueError(f"Unknown method: {method}")
            
            # Check if correlation is valid
            if np.isnan(corr) or np.isnan(pval):
                n_failed += 1
                continue
            
            # Store results for all vertices in this parcel
            rsa_correlations[parcel_mask] = corr
            if save_pvalues:
                rsa_pvalues[parcel_mask] = pval
            
            # Store per-parcel results
            parcel_results_list.append({
                'parcel_id': parcel_id,
                'parcel_name': parcel_name,
                'correlation': corr,
                'pvalue': pval,
                'n_vertices': n_parcel_vertices
            })
            
            n_computed += 1
            
        except Exception as e:
            n_failed += 1
            if n_failed <= 5:
                print(f"  Error at parcel {parcel_id}: {str(e)}")
            continue
    
    print(f"\nRSA computation complete!")
    print(f"  Successfully computed: {n_computed} / {len(unique_parcels)} ({100*n_computed/len(unique_parcels):.1f}%)")
    print(f"  Skipped (low variance): {n_skipped} ({100*n_skipped/len(unique_parcels):.1f}%)")
    print(f"  Failed (errors): {n_failed} ({100*n_failed/len(unique_parcels):.1f}%)")
    
    # Create DataFrame from results
    parcel_results = pd.DataFrame(parcel_results_list)
    
    if len(parcel_results) > 0:
        # Sort by correlation (descending)
        parcel_results = parcel_results.sort_values('correlation', ascending=False)
        
        print(f"\nValid correlation statistics:")
        print(f"  Range: [{parcel_results['correlation'].min():.4f}, {parcel_results['correlation'].max():.4f}]")
        print(f"  Mean: {parcel_results['correlation'].mean():.4f} ± {parcel_results['correlation'].std():.4f}")
        print(f"  Median: {parcel_results['correlation'].median():.4f}")
        
        n_sig_05 = (parcel_results['pvalue'] < 0.05).sum()
        n_sig_01 = (parcel_results['pvalue'] < 0.01).sum()
        n_sig_001 = (parcel_results['pvalue'] < 0.001).sum()
        
        print(f"\nSignificance (uncorrected p-values):")
        print(f"  p < 0.05:  {n_sig_05} / {len(parcel_results)} ({100*n_sig_05/len(parcel_results):.1f}%)")
        print(f"  p < 0.01:  {n_sig_01} / {len(parcel_results)} ({100*n_sig_01/len(parcel_results):.1f}%)")
        print(f"  p < 0.001: {n_sig_001} / {len(parcel_results)} ({100*n_sig_001/len(parcel_results):.1f}%)")
        print(f"\n  Note: These are uncorrected p-values.")
        print(f"    Consider FDR correction for {len(unique_parcels)} multiple comparisons.")
    else:
        print("\nWARNING: No valid correlations computed!")
        print("   Check your data for issues (constant values, NaNs, etc.)")
    
    # Save results
    if output_path:
        output_path = Path(output_path)
        
        # Save correlations
        np.save(output_path.with_suffix('.npy'), rsa_correlations)
        print(f"\nSaved correlations to {output_path.with_suffix('.npy')}")
        
        if save_pvalues:
            pval_file = output_path.with_name(output_path.stem + '_pvalues.npy')
            np.save(pval_file, rsa_pvalues)
            print(f"Saved p-values to {pval_file}")
        
        # Save per-parcel results as CSV
        if len(parcel_results) > 0:
            csv_file = output_path.with_name(output_path.stem + '_parcels.csv')
            parcel_results.to_csv(csv_file, index=False)
            print(f"Saved parcel results to {csv_file}")
        
        # Save as GIFTI metric for visualization
        darray = nib.gifti.GiftiDataArray(
            data=rsa_correlations.astype(np.float32),
            intent='NIFTI_INTENT_NONE',
            datatype='NIFTI_TYPE_FLOAT32'
        )
        
        meta = nib.gifti.GiftiMetaData()
        meta['Name'] = f"RSA_{method}"
        darray.meta = meta
        
        gii = nib.GiftiImage(darrays=[darray])
        nib.save(gii, str(output_path.with_suffix('.func.gii')))
        print(f"Saved GIFTI metric to {output_path.with_suffix('.func.gii')}")
    
    if save_pvalues:
        return rsa_correlations, rsa_pvalues
    else:
        return rsa_correlations

In [ ]:
# ============================================================================
# COMPLETE PIPELINE FUNCTION
# ============================================================================

def complete_rsa_pipeline(modality,parcel_labels, parcel_info, fmri_data, model_file, output_dir,
                          method='spearman'):
    """
    Complete Glasser parcellation RSA pipeline.
    
    Parameters:
    -----------
    modality : str
        Modality label ('joint', 'video', 'audio')
    parcel_labels : np.ndarray
        Parcel labels for each vertex
    parcel_info : dict
        Parcel name dictionary
    fmri_data : np.ndarray
        Preprocessed fMRI data (n_vertices, n_timepoints)
    model_file : str
        Model RDM or embeddings (.npy file)
    output_dir : str
        Output directory
    method : str
        'spearman' or 'pearson' or 'kendall'
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print("=" * 70)
    print("GLASSER PARCELLATION RSA PIPELINE")
    print("=" * 70)
    
    # Compute RSA map
    print("\nSTEP 1: Computing RSA correlation map...")
    print("-" * 70)
    
    rsa_output = output_dir / f"rsa_{modality}_glasser_{method}"
    
    rsa_correlations, rsa_pvalues = compute_rsa_map(
        fmri_data=fmri_data,
        parcel_labels=parcel_labels,
        parcel_info=parcel_info,
        model_rdm=model_file,
        output_path=str(rsa_output),
        method=method,
        save_pvalues=True,
        min_variance=1e-10
    )
    
    print("\n" + "=" * 70)
    print("PIPELINE COMPLETE!")
    print("=" * 70)
    print(f"Results saved in: {output_dir}")
    print(f"  RSA correlation map: {rsa_output}.func.gii")
    print(f"  RSA p-values: {rsa_output}_pvalues.npy")
    print(f"  Parcel results: {rsa_output}_parcels.csv")
    return rsa_correlations, rsa_pvalues

# Glasser RSA execution

## pe-av-small-16-frame_delay4s_bin2s

### Joint

In [ ]:
# Configuration
modality = 'joint'
DATA_DIR = Path("/home/amin/Research/Representation/Movie/data/Setareh")
HCP_DATA_DIR = DATA_DIR / "HCP Data"
OUTPUT_DIR = Path("/home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/pe-av-small-16-frame_delay4s_bin2s/joint")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Input fMRI files
mat_file_left = HCP_DATA_DIR / "notmean_left_Meanfile.mat"
mat_file_right = HCP_DATA_DIR / "notmean_right_Meanfile.mat"

# Glasser parcellation files (separate hemispheres)
glasser_label_left = HCP_DATA_DIR / "Q1-Q6_RelatedValidation210_LEFT.label.gii"
glasser_label_right = HCP_DATA_DIR / "Q1-Q6_RelatedValidation210_RIGHT.label.gii"

# Timing file
timing_csv = DATA_DIR / "Data/movie_timing.csv"

# Model files
model_file = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings/pe-av-small-16-frame/peav_small_16frame_av.npy")

# Parameters
RSA_METHOD = 'spearman'
TR = 1.0
BIN_SEC = 2.0
DELAY_SEC = 4.0

# ============================================================================
# STEP 0: Processing fMRI with timing/segmentation
# ============================================================================
print("=" * 70)
print("STEP 0: Processing fMRI with timing/segmentation")
print("=" * 70)

# Load Glasser parcellation (separate hemisphere files)
left_labels, right_labels, left_info, right_info = load_glasser_parcellation(
    glasser_label_left, glasser_label_right
)

# Load timing
timing_df = pd.read_csv(timing_csv)
print(f"\nTiming information loaded:")
print(timing_df)

# Process LEFT hemisphere
print("\n--- Processing LEFT hemisphere ---")
mat_data_left = sio.loadmat(mat_file_left)
fmri_left_raw = mat_data_left['Left_data']
if fmri_left_raw.shape[0] != 32492:
    fmri_left_raw = fmri_left_raw.T

print(f"Raw fMRI shape: {fmri_left_raw.shape}")
fmri_left_processed = process_all_video_segments(
    fmri_left_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC
)
print(f"Processed fMRI shape: {fmri_left_processed.shape}")

# Process RIGHT hemisphere
print("\n--- Processing RIGHT hemisphere ---")
mat_data_right = sio.loadmat(mat_file_right)
fmri_right_raw = mat_data_right['Right_data']
if fmri_right_raw.shape[0] != 32492:
    fmri_right_raw = fmri_right_raw.T

print(f"Raw fMRI shape: {fmri_right_raw.shape}")
fmri_right_processed = process_all_video_segments(
    fmri_right_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC
)
print(f"Processed fMRI shape: {fmri_right_processed.shape}")


# ============================================================================
# STEP 1: Run RSA pipeline for LEFT hemisphere
# ============================================================================
print("\n" + "=" * 70)
print("PROCESSING LEFT HEMISPHERE")
print("=" * 70)

rsa_corr_left, rsa_pval_left = complete_rsa_pipeline(modality=modality,
    parcel_labels=left_labels,
    parcel_info=left_info,
    fmri_data=fmri_left_processed,
    model_file=str(model_file),
    output_dir=str(OUTPUT_DIR / "left_hemisphere"),
    method=RSA_METHOD
)

# ============================================================================
# STEP 2: Run RSA pipeline for RIGHT hemisphere
# ============================================================================
print("\n" + "=" * 70)
print("PROCESSING RIGHT HEMISPHERE")
print("=" * 70)

rsa_corr_right, rsa_pval_right = complete_rsa_pipeline(modality=modality,
    parcel_labels=right_labels,
    parcel_info=right_info,
    fmri_data=fmri_right_processed,
    model_file=str(model_file),
    output_dir=str(OUTPUT_DIR / "right_hemisphere"),
    method=RSA_METHOD
)

# ============================================================================
# SUMMARY & STATISTICS
# ============================================================================
print("\n" + "=" * 70)
print("ALL PROCESSING COMPLETE!")
print("=" * 70)

print(f"\nOutput directory: {OUTPUT_DIR}")

print(f"\n{'='*70}")
print("LEFT HEMISPHERE RESULTS:")
print(f"{'='*70}")
print(f"  RSA correlation range: [{rsa_corr_left.min():.4f}, {rsa_corr_left.max():.4f}]")
print(f"  Mean correlation: {rsa_corr_left.mean():.4f} ± {rsa_corr_left.std():.4f}")
print(f"  Median correlation: {np.median(rsa_corr_left):.4f}")
print(f"  Significant vertices (p < 0.05): {(rsa_pval_left < 0.05).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.05).sum()/len(rsa_pval_left):.1f}%)")
print(f"  Significant vertices (p < 0.01): {(rsa_pval_left < 0.01).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.01).sum()/len(rsa_pval_left):.1f}%)")
print(f"  Significant vertices (p < 0.001): {(rsa_pval_left < 0.001).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.001).sum()/len(rsa_pval_left):.1f}%)")

print(f"\n{'='*70}")
print("RIGHT HEMISPHERE RESULTS:")
print(f"{'='*70}")
print(f"  RSA correlation range: [{rsa_corr_right.min():.4f}, {rsa_corr_right.max():.4f}]")
print(f"  Mean correlation: {rsa_corr_right.mean():.4f} ± {rsa_corr_right.std():.4f}")
print(f"  Median correlation: {np.median(rsa_corr_right):.4f}")
print(f"  Significant vertices (p < 0.05): {(rsa_pval_right < 0.05).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.05).sum()/len(rsa_pval_right):.1f}%)")
print(f"  Significant vertices (p < 0.01): {(rsa_pval_right < 0.01).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.01).sum()/len(rsa_pval_right):.1f}%)")
print(f"  Significant vertices (p < 0.001): {(rsa_pval_right < 0.001).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.001).sum()/len(rsa_pval_right):.1f}%)")

# ============================================================================
# VISUALIZATION COMMANDS
# ============================================================================
print(f"\n{'='*70}")
print(".func.gii results for wb_view visualization:")
print(f"{'='*70}")
print("\nLeft hemisphere:\n")
print(f"    {OUTPUT_DIR}/left_hemisphere/rsa_map_{RSA_METHOD}_glasser.func.gii")

print("\nRight hemisphere:\n")
print(f"    {OUTPUT_DIR}/right_hemisphere/rsa_map_{RSA_METHOD}_glasser.func.gii")

# ============================================================================
# SAVE COMBINED SUMMARY
# ============================================================================
summary_file = OUTPUT_DIR / "rsa_analysis_summary.txt"
with open(summary_file, 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("GLASSER PARCELLATION RSA ANALYSIS SUMMARY\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Analysis type: Glasser parcellation-based\n")
    f.write(f"RSA method: {RSA_METHOD}\n\n")
    
    f.write("LEFT HEMISPHERE:\n")
    f.write(f"  Mean correlation: {rsa_corr_left.mean():.4f}\n")
    f.write(f"  Significant (p<0.05): {(rsa_pval_left < 0.05).sum()} vertices\n\n")
    
    f.write("RIGHT HEMISPHERE:\n")
    f.write(f"  Mean correlation: {rsa_corr_right.mean():.4f}\n")
    f.write(f"  Significant (p<0.05): {(rsa_pval_right < 0.05).sum()} vertices\n")

print(f"\nSummary saved to: {summary_file}")

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE! 🎉")
print("=" * 70)

STEP 0: Processing fMRI with timing/segmentation
Loading Glasser parcellation from separate hemisphere files...

Left hemisphere: /home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_LEFT.label.gii
Right hemisphere: /home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_RIGHT.label.gii

Left hemisphere: 180 parcels
  Parcel IDs range: 181 to 360
  Total vertices: 32492

Right hemisphere: 180 parcels
  Parcel IDs range: 1 to 180
  Total vertices: 32492

Timing information loaded:
   video_id  onset_sec  end_sec  duration_sec
0    video1         20      232           212
1    video2        284      504           220
2    video3        526      712           186
3    video4        735      797            62
4    video5        818      900            82
5    video6        920     1146           226
6    video7       1167     1425           258
7    video8       1446     1694           248
8    video9       1716     179

### audio

In [ ]:
# Configuration
modality = 'audio'
DATA_DIR = Path("/home/amin/Research/Representation/Movie/data/Setareh")
HCP_DATA_DIR = DATA_DIR / "HCP Data"
OUTPUT_DIR = Path("/home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/pe-av-small-16-frame_delay4s_bin2s/audio")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Input fMRI files
mat_file_left = HCP_DATA_DIR / "notmean_left_Meanfile.mat"
mat_file_right = HCP_DATA_DIR / "notmean_right_Meanfile.mat"

# Glasser parcellation files (separate hemispheres)
glasser_label_left = HCP_DATA_DIR / "Q1-Q6_RelatedValidation210_LEFT.label.gii"
glasser_label_right = HCP_DATA_DIR / "Q1-Q6_RelatedValidation210_RIGHT.label.gii"

# Timing file
timing_csv = DATA_DIR / "Data/movie_timing.csv"

# Model files
model_file = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings/pe-av-small-16-frame/peav_small_16frame_a.npy")

# Parameters
RSA_METHOD = 'spearman'
TR = 1.0
BIN_SEC = 2.0
DELAY_SEC = 4.0

# ============================================================================
# STEP 0: Processing fMRI with timing/segmentation
# ============================================================================
print("=" * 70)
print("STEP 0: Processing fMRI with timing/segmentation")
print("=" * 70)

# Load Glasser parcellation (separate hemisphere files)
left_labels, right_labels, left_info, right_info = load_glasser_parcellation(
    glasser_label_left, glasser_label_right
)

# Load timing
timing_df = pd.read_csv(timing_csv)
print(f"\nTiming information loaded:")
print(timing_df)

# Process LEFT hemisphere
print("\n--- Processing LEFT hemisphere ---")
mat_data_left = sio.loadmat(mat_file_left)
fmri_left_raw = mat_data_left['Left_data']
if fmri_left_raw.shape[0] != 32492:
    fmri_left_raw = fmri_left_raw.T

print(f"Raw fMRI shape: {fmri_left_raw.shape}")
fmri_left_processed = process_all_video_segments(
    fmri_left_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC
)
print(f"Processed fMRI shape: {fmri_left_processed.shape}")

# Process RIGHT hemisphere
print("\n--- Processing RIGHT hemisphere ---")
mat_data_right = sio.loadmat(mat_file_right)
fmri_right_raw = mat_data_right['Right_data']
if fmri_right_raw.shape[0] != 32492:
    fmri_right_raw = fmri_right_raw.T

print(f"Raw fMRI shape: {fmri_right_raw.shape}")
fmri_right_processed = process_all_video_segments(
    fmri_right_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC
)
print(f"Processed fMRI shape: {fmri_right_processed.shape}")


# ============================================================================
# STEP 1: Run RSA pipeline for LEFT hemisphere
# ============================================================================
print("\n" + "=" * 70)
print("PROCESSING LEFT HEMISPHERE")
print("=" * 70)

rsa_corr_left, rsa_pval_left = complete_rsa_pipeline(modality=modality,
    parcel_labels=left_labels,
    parcel_info=left_info,
    fmri_data=fmri_left_processed,
    model_file=str(model_file),
    output_dir=str(OUTPUT_DIR / "left_hemisphere"),
    method=RSA_METHOD
)

# ============================================================================
# STEP 2: Run RSA pipeline for RIGHT hemisphere
# ============================================================================
print("\n" + "=" * 70)
print("PROCESSING RIGHT HEMISPHERE")
print("=" * 70)

rsa_corr_right, rsa_pval_right = complete_rsa_pipeline(modality =modality,
    parcel_labels=right_labels,
    parcel_info=right_info,
    fmri_data=fmri_right_processed,
    model_file=str(model_file),
    output_dir=str(OUTPUT_DIR / "right_hemisphere"),
    method=RSA_METHOD
)

# ============================================================================
# SUMMARY & STATISTICS
# ============================================================================
print("\n" + "=" * 70)
print("ALL PROCESSING COMPLETE!")
print("=" * 70)

print(f"\nOutput directory: {OUTPUT_DIR}")

print(f"\n{'='*70}")
print("LEFT HEMISPHERE RESULTS:")
print(f"{'='*70}")
print(f"  RSA correlation range: [{rsa_corr_left.min():.4f}, {rsa_corr_left.max():.4f}]")
print(f"  Mean correlation: {rsa_corr_left.mean():.4f} ± {rsa_corr_left.std():.4f}")
print(f"  Median correlation: {np.median(rsa_corr_left):.4f}")
print(f"  Significant vertices (p < 0.05): {(rsa_pval_left < 0.05).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.05).sum()/len(rsa_pval_left):.1f}%)")
print(f"  Significant vertices (p < 0.01): {(rsa_pval_left < 0.01).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.01).sum()/len(rsa_pval_left):.1f}%)")
print(f"  Significant vertices (p < 0.001): {(rsa_pval_left < 0.001).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.001).sum()/len(rsa_pval_left):.1f}%)")

print(f"\n{'='*70}")
print("RIGHT HEMISPHERE RESULTS:")
print(f"{'='*70}")
print(f"  RSA correlation range: [{rsa_corr_right.min():.4f}, {rsa_corr_right.max():.4f}]")
print(f"  Mean correlation: {rsa_corr_right.mean():.4f} ± {rsa_corr_right.std():.4f}")
print(f"  Median correlation: {np.median(rsa_corr_right):.4f}")
print(f"  Significant vertices (p < 0.05): {(rsa_pval_right < 0.05).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.05).sum()/len(rsa_pval_right):.1f}%)")
print(f"  Significant vertices (p < 0.01): {(rsa_pval_right < 0.01).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.01).sum()/len(rsa_pval_right):.1f}%)")
print(f"  Significant vertices (p < 0.001): {(rsa_pval_right < 0.001).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.001).sum()/len(rsa_pval_right):.1f}%)")

# ============================================================================
# VISUALIZATION COMMANDS
# ============================================================================
print(f"\n{'='*70}")
print(".func.gii results for wb_view visualization:")
print(f"{'='*70}")
print("\nLeft hemisphere:\n")
print(f"    {OUTPUT_DIR}/left_hemisphere/rsa_map_{RSA_METHOD}_glasser.func.gii")

print("\nRight hemisphere:\n")
print(f"    {OUTPUT_DIR}/right_hemisphere/rsa_map_{RSA_METHOD}_glasser.func.gii")

# ============================================================================
# SAVE COMBINED SUMMARY
# ============================================================================
summary_file = OUTPUT_DIR / "rsa_analysis_summary.txt"
with open(summary_file, 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("GLASSER PARCELLATION RSA ANALYSIS SUMMARY\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Analysis type: Glasser parcellation-based\n")
    f.write(f"RSA method: {RSA_METHOD}\n\n")
    
    f.write("LEFT HEMISPHERE:\n")
    f.write(f"  Mean correlation: {rsa_corr_left.mean():.4f}\n")
    f.write(f"  Significant (p<0.05): {(rsa_pval_left < 0.05).sum()} vertices\n\n")
    
    f.write("RIGHT HEMISPHERE:\n")
    f.write(f"  Mean correlation: {rsa_corr_right.mean():.4f}\n")
    f.write(f"  Significant (p<0.05): {(rsa_pval_right < 0.05).sum()} vertices\n")

print(f"\nSummary saved to: {summary_file}")

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE! 🎉")
print("=" * 70)

STEP 0: Processing fMRI with timing/segmentation
Loading Glasser parcellation from separate hemisphere files...

Left hemisphere: /home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_LEFT.label.gii
Right hemisphere: /home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_RIGHT.label.gii

Left hemisphere: 180 parcels
  Parcel IDs range: 181 to 360
  Total vertices: 32492

Right hemisphere: 180 parcels
  Parcel IDs range: 1 to 180
  Total vertices: 32492

Timing information loaded:
   video_id  onset_sec  end_sec  duration_sec
0    video1         20      232           212
1    video2        284      504           220
2    video3        526      712           186
3    video4        735      797            62
4    video5        818      900            82
5    video6        920     1146           226
6    video7       1167     1425           258
7    video8       1446     1694           248
8    video9       1716     179

### video

In [ ]:
#### Configuration
modality = 'video'
DATA_DIR = Path("/home/amin/Research/Representation/Movie/data/Setareh")
HCP_DATA_DIR = DATA_DIR / "HCP Data"
OUTPUT_DIR = Path("/home/amin/Research/Representation/Movie/outputs/glasser_rsa_output/pe-av-small-16-frame_delay4s_bin2s/video")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Input fMRI files
mat_file_left = HCP_DATA_DIR / "notmean_left_Meanfile.mat"
mat_file_right = HCP_DATA_DIR / "notmean_right_Meanfile.mat"

# Glasser parcellation files (separate hemispheres)
glasser_label_left = HCP_DATA_DIR / "Q1-Q6_RelatedValidation210_LEFT.label.gii"
glasser_label_right = HCP_DATA_DIR / "Q1-Q6_RelatedValidation210_RIGHT.label.gii"

# Timing file
timing_csv = DATA_DIR / "Data/movie_timing.csv"

# Model files
model_file = Path("/home/amin/Research/Representation/Movie/outputs/model_embeddings/pe-av-small-16-frame/peav_small_16frame_v.npy")

# Parameters
RSA_METHOD = 'spearman'
TR = 1.0
BIN_SEC = 2.0
DELAY_SEC = 4.0

# ============================================================================
# STEP 0: Processing fMRI with timing/segmentation
# ============================================================================
print("=" * 70)
print("STEP 0: Processing fMRI with timing/segmentation")
print("=" * 70)

# Load Glasser parcellation (separate hemisphere files)
left_labels, right_labels, left_info, right_info = load_glasser_parcellation(
    glasser_label_left, glasser_label_right
)

# Load timing
timing_df = pd.read_csv(timing_csv)
print(f"\nTiming information loaded:")
print(timing_df)

# Process LEFT hemisphere
print("\n--- Processing LEFT hemisphere ---")
mat_data_left = sio.loadmat(mat_file_left)
fmri_left_raw = mat_data_left['Left_data']
if fmri_left_raw.shape[0] != 32492:
    fmri_left_raw = fmri_left_raw.T

print(f"Raw fMRI shape: {fmri_left_raw.shape}")
fmri_left_processed = process_all_video_segments(
    fmri_left_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC
)
print(f"Processed fMRI shape: {fmri_left_processed.shape}")

# Process RIGHT hemisphere
print("\n--- Processing RIGHT hemisphere ---")
mat_data_right = sio.loadmat(mat_file_right)
fmri_right_raw = mat_data_right['Right_data']
if fmri_right_raw.shape[0] != 32492:
    fmri_right_raw = fmri_right_raw.T

print(f"Raw fMRI shape: {fmri_right_raw.shape}")
fmri_right_processed = process_all_video_segments(
    fmri_right_raw, timing_df, TR=TR, bin_sec=BIN_SEC, delay_sec=DELAY_SEC
)
print(f"Processed fMRI shape: {fmri_right_processed.shape}")


# ============================================================================
# STEP 1: Run RSA pipeline for LEFT hemisphere
# ============================================================================
print("\n" + "=" * 70)
print("PROCESSING LEFT HEMISPHERE")
print("=" * 70)

rsa_corr_left, rsa_pval_left = complete_rsa_pipeline(modality= modality, 
    parcel_labels=left_labels,
    parcel_info=left_info,
    fmri_data=fmri_left_processed,
    model_file=str(model_file),
    output_dir=str(OUTPUT_DIR / "left_hemisphere"),
    method=RSA_METHOD
)

# ============================================================================
# STEP 2: Run RSA pipeline for RIGHT hemisphere
# ============================================================================
print("\n" + "=" * 70)
print("PROCESSING RIGHT HEMISPHERE")
print("=" * 70)

rsa_corr_right, rsa_pval_right = complete_rsa_pipeline(modality= modality, 
    parcel_labels=right_labels,
    parcel_info=right_info,
    fmri_data=fmri_right_processed,
    model_file=str(model_file),
    output_dir=str(OUTPUT_DIR / "right_hemisphere"),
    method=RSA_METHOD
)

# ============================================================================
# SUMMARY & STATISTICS
# ============================================================================
print("\n" + "=" * 70)
print("ALL PROCESSING COMPLETE!")
print("=" * 70)

print(f"\nOutput directory: {OUTPUT_DIR}")

print(f"\n{'='*70}")
print("LEFT HEMISPHERE RESULTS:")
print(f"{'='*70}")
print(f"  RSA correlation range: [{rsa_corr_left.min():.4f}, {rsa_corr_left.max():.4f}]")
print(f"  Mean correlation: {rsa_corr_left.mean():.4f} ± {rsa_corr_left.std():.4f}")
print(f"  Median correlation: {np.median(rsa_corr_left):.4f}")
print(f"  Significant vertices (p < 0.05): {(rsa_pval_left < 0.05).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.05).sum()/len(rsa_pval_left):.1f}%)")
print(f"  Significant vertices (p < 0.01): {(rsa_pval_left < 0.01).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.01).sum()/len(rsa_pval_left):.1f}%)")
print(f"  Significant vertices (p < 0.001): {(rsa_pval_left < 0.001).sum():,} / {len(rsa_pval_left):,} ({100*(rsa_pval_left < 0.001).sum()/len(rsa_pval_left):.1f}%)")

print(f"\n{'='*70}")
print("RIGHT HEMISPHERE RESULTS:")
print(f"{'='*70}")
print(f"  RSA correlation range: [{rsa_corr_right.min():.4f}, {rsa_corr_right.max():.4f}]")
print(f"  Mean correlation: {rsa_corr_right.mean():.4f} ± {rsa_corr_right.std():.4f}")
print(f"  Median correlation: {np.median(rsa_corr_right):.4f}")
print(f"  Significant vertices (p < 0.05): {(rsa_pval_right < 0.05).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.05).sum()/len(rsa_pval_right):.1f}%)")
print(f"  Significant vertices (p < 0.01): {(rsa_pval_right < 0.01).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.01).sum()/len(rsa_pval_right):.1f}%)")
print(f"  Significant vertices (p < 0.001): {(rsa_pval_right < 0.001).sum():,} / {len(rsa_pval_right):,} ({100*(rsa_pval_right < 0.001).sum()/len(rsa_pval_right):.1f}%)")

# ============================================================================
# VISUALIZATION COMMANDS
# ============================================================================
print(f"\n{'='*70}")
print(".func.gii results for wb_view visualization:")
print(f"{'='*70}")
print("\nLeft hemisphere:\n")
print(f"    {OUTPUT_DIR}/left_hemisphere/rsa_map_{RSA_METHOD}_glasser.func.gii")

print("\nRight hemisphere:\n")
print(f"    {OUTPUT_DIR}/right_hemisphere/rsa_map_{RSA_METHOD}_glasser.func.gii")

# ============================================================================
# SAVE COMBINED SUMMARY
# ============================================================================
summary_file = OUTPUT_DIR / "rsa_analysis_summary.txt"
with open(summary_file, 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("GLASSER PARCELLATION RSA ANALYSIS SUMMARY\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Analysis type: Glasser parcellation-based\n")
    f.write(f"RSA method: {RSA_METHOD}\n\n")
    
    f.write("LEFT HEMISPHERE:\n")
    f.write(f"  Mean correlation: {rsa_corr_left.mean():.4f}\n")
    f.write(f"  Significant (p<0.05): {(rsa_pval_left < 0.05).sum()} vertices\n\n")
    
    f.write("RIGHT HEMISPHERE:\n")
    f.write(f"  Mean correlation: {rsa_corr_right.mean():.4f}\n")
    f.write(f"  Significant (p<0.05): {(rsa_pval_right < 0.05).sum()} vertices\n")

print(f"\nSummary saved to: {summary_file}")

print("\n" + "=" * 70)
print("ANALYSIS COMPLETE! 🎉")
print("=" * 70)

STEP 0: Processing fMRI with timing/segmentation
Loading Glasser parcellation from separate hemisphere files...

Left hemisphere: /home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_LEFT.label.gii
Right hemisphere: /home/amin/Research/Representation/Movie/data/Setareh/HCP Data/Q1-Q6_RelatedValidation210_RIGHT.label.gii

Left hemisphere: 180 parcels
  Parcel IDs range: 181 to 360
  Total vertices: 32492

Right hemisphere: 180 parcels
  Parcel IDs range: 1 to 180
  Total vertices: 32492

Timing information loaded:
   video_id  onset_sec  end_sec  duration_sec
0    video1         20      232           212
1    video2        284      504           220
2    video3        526      712           186
3    video4        735      797            62
4    video5        818      900            82
5    video6        920     1146           226
6    video7       1167     1425           258
7    video8       1446     1694           248
8    video9       1716     179